# Tratamento Inicial dos Datasets

Este notebook aplica o mesmo tratamento inicial do projeto de regressão logística:
- Cria janelas futuras (3, 7, 15, 30 dias) para o preço de fechamento
- Gera rótulos binários (alta/baixa) para cada horizonte
- Remove linhas com NaN geradas pelos shifts
- Salva os arquivos tratados em `Datasets_processados`

Observação: não adiciona variável dummy de período.

In [ ]:
import os
from pathlib import Path
import pandas as pd

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 1. Configuração de pastas

In [ ]:
BASE_DIR = Path('..').resolve()  # Base/
DATASETS_DIR = BASE_DIR / 'Datasets'
OUTPUT_DIR = DATASETS_DIR / 'Datasets_processados'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Pasta de entrada:', DATASETS_DIR)
print('Pasta de saída:', OUTPUT_DIR)

Pasta de entrada: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/Datasets
Pasta de saída: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/Datasets_processados


## 2. Função de tratamento

In [ ]:
def tratar_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Criação de janelas futuras para o preço de fechamento
    df = df.copy()
    df['Close_3d_fut'] = df['Close'].shift(-3)
    df['Close_7d_fut'] = df['Close'].shift(-7)
    df['Close_15d_fut'] = df['Close'].shift(-15)
    df['Close_30d_fut'] = df['Close'].shift(-30)

    # Remove linhas com NaN nas janelas futuras
    df = df.dropna(subset=['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut'])

    # Converte coluna de data e define índice, se existir
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date')

    # Cria rótulos binários (1 = alta, 0 = baixa/estável)
    df['target_3d'] = (df['Close_3d_fut'] > df['Close']).astype(int)
    df['target_7d'] = (df['Close_7d_fut'] > df['Close']).astype(int)
    df['target_15d'] = (df['Close_15d_fut'] > df['Close']).astype(int)
    df['target_30d'] = (df['Close_30d_fut'] > df['Close']).astype(int)

    return df

## 3. Processamento em lote

In [ ]:
csv_files = sorted(DATASETS_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum CSV encontrado em {DATASETS_DIR}')

print(f'Arquivos encontrados: {len(csv_files)}')
for f in csv_files:
    print('-', f.name)

Arquivos encontrados: 3
- AGRO3.csv
- SLCE3.csv
- SOJA3.csv


In [ ]:
for file_path in csv_files:
    print('' + '=' * 80)
    print(f'Processando: {file_path.name}')

    df = pd.read_csv(file_path)
    print('Shape original:', df.shape)

    # Validação mínima
    required_cols = {'Close'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        print(f'ERRO: colunas ausentes {missing_cols}. Arquivo ignorado.')
        continue

    df_tratado = tratar_dataset(df)
    print('Shape tratado:', df_tratado.shape)

    # Salvar arquivo tratado
    output_name = file_path.stem + '_tratado.csv'
    output_path = OUTPUT_DIR / output_name
    df_tratado.to_csv(output_path, index=True)
    print('Salvo em:', output_path)

Processando: AGRO3.csv
Shape original: (1740, 6)
Shape tratado: (1710, 14)
Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/Datasets_processados/AGRO3_tratado.csv
Processando: SLCE3.csv
Shape original: (1738, 6)
Shape tratado: (1708, 13)
Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/Datasets_processados/SLCE3_tratado.csv
Processando: SOJA3.csv
Shape original: (919, 6)
Shape tratado: (889, 14)
Salvo em: /home/gabriel/Documents/GIT/Analise-comparativa-de-modelos-de-machine-learning---Agro-Brasil---TCC/Datasets_processados/SOJA3_tratado.csv


## 4. Resumo final

In [ ]:
processed_files = sorted(OUTPUT_DIR.glob('*_tratado.csv'))
print('Arquivos tratados gerados:')
for f in processed_files:
    print('-', f.name)

Arquivos tratados gerados:
- AGRO3_tratado.csv
- SLCE3_tratado.csv
- SOJA3_tratado.csv
